In [ ]:
import os
import kagglehub
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder

In [ ]:
# 1. Загрузка данных
path = kagglehub.competition_download("playground-series-s6e11")
train_df = pd.read_csv(os.path.join(path, "train.csv"))
test_df = pd.read_csv(os.path.join(path, "test.csv"))

In [ ]:
# 2. Отделяем признаки (X) от ответов (y) и делим на обучение и проверку
X = train_df.drop(columns=["id", "is_fraud"])
y = train_df["is_fraud"]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
# 3. Простая подготовка данных: переводим текст и True/False в обычные числа (0, 1, 2...)
cat_cols = X.select_dtypes(include=["object", "bool"]).columns.tolist()

encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)

# Заменяем текстовые колонки на числа прямо в таблицах
X_train[cat_cols] = encoder.fit_transform(X_train[cat_cols].astype(str))
X_val[cat_cols] = encoder.transform(X_val[cat_cols].astype(str))

In [ ]:
# 4. Создаем и обучаем Случайный лес!
rf_model = RandomForestClassifier(
    n_estimators=100,  # Создаем лес из 100 деревьев
    max_depth=12,  # Ограничиваем глубину деревьев (чтобы училось быстро и не зубрило)
    class_weight="balanced",  # Модель САМА учтет, что мошенников всего 1.67%!
    n_jobs=-1,  # Задействуем все ядра процессора для максимальной скорости
    random_state=42,
)

print("Обучаем Случайный лес...")
rf_model.fit(X_train, y_train)

In [ ]:
# 5. Проверяем качество (ROC-AUC)
# ВАЖНО: метод .predict_proba() выдает вероятности для обоих классов [вероятность_0, вероятность_1].
# Нам нужна вероятность мошенничества (класс 1), поэтому берем второй столбец [:, 1]
val_preds = rf_model.predict_proba(X_val)[:, 1]
score = roc_auc_score(y_val, val_preds)

print(f"Качество Случайного леса (ROC-AUC): {score:.4f}")

In [ ]:
# 6. Предсказание для тестового файла (Submission)
X_test = test_df.drop(columns=["id"])
X_test[cat_cols] = encoder.transform(X_test[cat_cols].astype(str))

test_preds = rf_model.predict_proba(X_test)[:, 1]

submission = pd.DataFrame({"id": test_df["id"], "is_fraud": test_preds})
submission.to_csv("rf_submission.csv", index=False)
print("Готово! Файл rf_submission.csv сохранен.")

Обучаем Случайный лес...
Качество Случайного леса (ROC-AUC): 0.9227
Готово! Файл rf_submission.csv сохранен.
